# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [1]:
import os
import sys

repo_url  = "https://github.com/Seongha-parkk/2026-HYU-AUE8088-PA2.git"
repo_name = "2026-HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    # 런타임이 살아있고 이미 클론된 경우 최신 코드로 업데이트
    !git -C /content/{repo_name} pull

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Already up to date.
/content/2026-HYU-AUE8088-PA2
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
from src.models.swin import SwinTiny

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
import wandb; wandb.login()   # API key 입력

# wandb 설정 — 비활성화하려면 None
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level2"]

In [6]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋이 이미 존재합니다 → ../data/set_a


In [7]:
import subprocess, sys

# 출처: Touvron et al., DeiT (ICML 2021)
# https://github.com/facebookresearch/deit
# 체크포인트: deit_small_patch16_224-cd65a155.pth (ImageNet-1k pretrained)
CKPT_PATH = "../deit_s16.pth"

if not os.path.exists(CKPT_PATH):
    print("DeiT-S/16 체크포인트 다운로드 중...")
    subprocess.run([
        "wget", "-q",
        "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth",
        "-O", CKPT_PATH,
    ], check=True)
    print("완료")


def remap_deit(state_dict: dict) -> dict:
    """DeiT state_dict 키를 학생 ViT 구현 키로 변환.

    주요 차이:
      DeiT  mlp.fc1.*  →  학생 mlp.0.*  (nn.Sequential 인덱스)
      DeiT  mlp.fc2.*  →  학생 mlp.3.*
      head.*           →  스킵 (MultiTaskHead 는 랜덤 초기화 유지)
    """
    new_sd = {}
    for k, v in state_dict.items():
        if k.startswith("head."):
            continue
        k = k.replace(".mlp.fc1.", ".mlp.0.")
        k = k.replace(".mlp.fc2.", ".mlp.3.")
        new_sd[k] = v
    return new_sd


USE_PRETRAINED = False
model = vit_small_patch16_224().to(device)

if USE_PRETRAINED:
    raw = torch.load(CKPT_PATH, map_location="cpu")
    sd  = raw.get("model", raw)          # DeiT 체크포인트는 'model' 키 안에 있음
    remapped = remap_deit(sd)
    missing, unexpected = model.load_state_dict(remapped, strict=False)
    print(f"로드 완료 — missing: {len(missing)}, unexpected: {len(unexpected)}")
    print(f"  missing keys (head.*만 나와야 정상): {missing}")


DeiT-S/16 체크포인트 다운로드 중...
완료
로드 완료 — missing: 6, unexpected: 0
  missing keys (head.*만 나와야 정상): ['head.weather.weight', 'head.weather.bias', 'head.scene.weight', 'head.scene.bias', 'head.timeofday.weight', 'head.timeofday.bias']


In [8]:
epochs = 30
optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level2-vit_s16{'-pretrained' if USE_PRETRAINED else ''}",
    config={
        "backbone": "vit_s16", "pretrained": USE_PRETRAINED,
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + ["vit_s16"],
)
trainer = MultiTaskTrainer(model, optim, sched, losses, device, TrainConfig(epochs=epochs), logger=logger)

history = trainer.fit(train_loader, val_loader)

val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
logger.finish()

os.makedirs("../checkpoints", exist_ok=True)
torch.save({"state_dict": model.state_dict(), "history": history},
           "../checkpoints/level2_vit_s16.pth")
print("저장 완료 → ../checkpoints/level2_vit_s16.pth")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=2.1193  val_avg_MF1=0.4835  per={'weather': 0.27835494517800846, 'scene': 0.504200395600375, 'timeofday': 0.6679674001323311}


[epoch 02/30] train_loss=1.6880  val_avg_MF1=0.5661  per={'weather': 0.4256262875516745, 'scene': 0.44193375436887056, 'timeofday': 0.830720798957881}


[epoch 03/30] train_loss=1.5569  val_avg_MF1=0.5491  per={'weather': 0.3977259396686971, 'scene': 0.40887084162788834, 'timeofday': 0.8408163806455492}


[epoch 04/30] train_loss=1.4790  val_avg_MF1=0.6394  per={'weather': 0.4732090535464967, 'scene': 0.5883186747502348, 'timeofday': 0.8567903888705262}


...

[epoch 28/30] train_loss=0.0227  val_avg_MF1=0.6646  per={'weather': 0.5500380695173263, 'scene': 0.6454000240144198, 'timeofday': 0.7984303922188136}


[epoch 29/30] train_loss=0.0246  val_avg_MF1=0.6591  per={'weather': 0.540228248387593, 'scene': 0.6355295962166191, 'timeofday': 0.8016072547559087}


[epoch 30/30] train_loss=0.0209  val_avg_MF1=0.6589  per={'weather': 0.540228248387593, 'scene': 0.6348619873664315, 'timeofday': 0.8016072547559087}


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▇▆▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▄▃▇▄▆▇▇▇▇▇█▇▇▇▇▆▇▇▆▇█▇▇████▇▇
val/mf1_scene,▄▂▁▆▂▅▇▆█▆▇▇█▇▇▇▆▆▇▆▆██▆███▇▇▇
val/mf1_timeofday,▁▇▇█▇▇▇▇▆▆▅▆▆▆▆▄▅▅▅▆▆▆▅▆▆▆▆▆▆▆
val/mf1_weather,▁▄▄▆▅▆▆▇▆▇▇█▆▆▇▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇
epoch,30
lr,0
train/loss,0.02087
val/avg_macro_f1,0.6589


저장 완료 → ../checkpoints/level2_vit_s16.pth


In [ ]:
# Swin-Tiny ImageNet pretrained weight 로드
# 출처: Liu et al., Swin Transformer (ICCV 2021)
# https://github.com/microsoft/Swin-Transformer
SWIN_CKPT_PATH = "../swin_tiny.pth"

if not os.path.exists(SWIN_CKPT_PATH):
    print("Swin-Tiny 체크포인트 다운로드 중...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth",
        "-O", SWIN_CKPT_PATH,
    ], check=True)
    print("완료")


def remap_swin(state_dict: dict) -> dict:
    """Microsoft Swin 체크포인트 키를 학생 구현 키로 변환.

    주요 차이:
      layers.X.*  →  stages.X.*   (ModuleList 이름)
      mlp.fc1.*   →  mlp.0.*      (nn.Sequential 인덱스)
      mlp.fc2.*   →  mlp.3.*
      head.*      →  스킵 (MultiTaskHead 랜덤 초기화 유지)
      attn_mask   →  스킵 (동적으로 계산)
    """
    new_sd = {}
    for k, v in state_dict.items():
        if k.startswith("head.") or "attn_mask" in k:
            continue
        k = k.replace("layers.", "stages.")
        k = k.replace(".mlp.fc1.", ".mlp.0.")
        k = k.replace(".mlp.fc2.", ".mlp.3.")
        new_sd[k] = v
    return new_sd


SWIN_PRETRAINED = True
set_seed(SEED, deterministic=True)
swin_model = SwinTiny().to(device)

if SWIN_PRETRAINED:
    raw = torch.load(SWIN_CKPT_PATH, map_location="cpu")
    sd = raw.get("model", raw)
    remapped = remap_swin(sd)
    missing, unexpected = swin_model.load_state_dict(remapped, strict=False)
    print(f"로드 완료 — missing: {len(missing)}, unexpected: {len(unexpected)}")
    print(f"  missing keys (head.*만 나와야 정상): {missing}")

EOFError: 

In [ ]:
# Swin-Tiny 학습
swin_epochs = 30
swin_optim = torch.optim.AdamW(swin_model.parameters(), lr=5e-4, weight_decay=5e-2)
swin_sched = torch.optim.lr_scheduler.CosineAnnealingLR(swin_optim, T_max=swin_epochs)
swin_losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

swin_logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level2-swin_tiny{'-pretrained' if SWIN_PRETRAINED else ''}",
    config={
        "backbone": "swin_tiny", "pretrained": SWIN_PRETRAINED,
        "epochs": swin_epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + ["swin_tiny"],
)
swin_trainer = MultiTaskTrainer(swin_model, swin_optim, swin_sched, swin_losses, device,
                                TrainConfig(epochs=swin_epochs), logger=swin_logger)

swin_history = swin_trainer.fit(train_loader, val_loader)

val_pred, _, val_tgt, _ = collect_predictions(swin_model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
for a in ATTRIBUTES:
    swin_logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
swin_logger.finish()

os.makedirs("../checkpoints", exist_ok=True)
torch.save({"state_dict": swin_model.state_dict(), "history": swin_history},
           "../checkpoints/level2_swin_tiny.pth")
print("저장 완료 → ../checkpoints/level2_swin_tiny.pth")

## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.